<a href="https://colab.research.google.com/github/t6niskoppel/Optimization-for-Robot-Motion-Planning-and-Control-assignment3/blob/main/assignment_3_tonis_koppel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3 — MPC Navigation in Cluttered Environments

Differential-drive robot navigating from LiDAR perception, using trajectory
optimization at every control loop as **Model Predictive Control** feedback.

Four planners (selected via `planner_type`), all implemented in `planner.py`:

- **`gd`** — plain gradient descent
- **`nesterov`** — Nesterov accelerated gradient
- **`cem`** — Cross-Entropy Method
- **`hybrid`** — CEM followed by Adam refinement

## Setup

In [ ]:
!pip install ir-sim[all]
!pip install open3d

In [ ]:
!git clone https://github.com/t6niskoppel/Optimization-for-Robot-Motion-Planning-and-Control-assignment3.git opt_irsim
%cd opt_irsim

## 1. Visualize each planner on `obstacle_world`

Each run saves an animation to `animation/<planner_type>/`.

In [ ]:
from test import run_episode, benchmark, barn_worlds

planners = ["gd", "nesterov", "cem", "hybrid"]
for pt in planners:
    m = run_episode("barn_envs/barn_43.yaml", planner_type=pt, save_ani=True)
    print(m)

### Saved animations

In [ ]:
import glob, os
from IPython.display import Image, display

for pt in planners:
    for g in sorted(glob.glob(f"animation/{pt}/*.gif")):
        print(pt, os.path.basename(g))
        display(Image(filename=g))

## 2. Benchmark across BARN environments

Run every planner on a set of barn worlds (rendering disabled) and report,
per planner: **success rate** (reached goal *and* no collision),
**collision rate**, **time-to-reach-goal**, **min clearance** to obstacles,
and mean **solve time per MPC step**.

In [ ]:
results = benchmark(barn_worlds(300))

### Metrics table

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
summary = df.groupby('planner').agg(
    success_rate=('success', 'mean'),
    collision_rate=('collided', 'mean'),
    mean_time_to_goal=('time_to_goal', 'mean'),
    mean_min_clearance=('min_clearance', 'mean'),
    mean_solve_ms=('solve_ms', 'mean'),
    n=('success', 'count'),
)
summary['success_rate'] = (summary['success_rate'] * 100).round(0)
summary['collision_rate'] = (summary['collision_rate'] * 100).round(0)
summary = summary.round(2)
summary